# Clase 8: programa tu propio SVM lineal desde cero

En el capítulo anterior descubriste la intuición de la máquina de vectores de soporte (SVM, por sus siglas en inglés *Support Vector Machine*): un algoritmo que busca la línea, o el plano, que separa dos grupos de puntos dejando el mayor margen posible a cada lado. También la entrenaste en dos líneas con `SVC`, la clase de scikit-learn pensada para eso.

Hoy abres el capó. Vas a construir tu propia clase SVM en Python, sin depender de ninguna librería de machine learning. Programas tres piezas: un método `fit` que busca los mejores valores de `w` (el vector de pesos) y `b` (el sesgo) probando y descartando opciones, un método `predict` que clasifica puntos nuevos, y una visualización con Matplotlib que dibuja los puntos, el hiperplano y los márgenes.

¿Para qué sirve esto si scikit-learn ya lo resuelve? Porque programar el algoritmo a mano te obliga a entender qué significa realmente cada línea de la fórmula que viste en el capítulo anterior. Cuando más adelante uses cualquier librería, sabes qué pasa por dentro y puedes diagnosticar problemas con criterio propio, no solo por ensayo y error.

Una advertencia importante: el método que programas aquí busca `w` y `b` probando valores cada vez más pequeños y comprobando restricciones, sin descenso de gradiente ni solucionadores de optimización externos. Es un método sencillo, pensado para aprender, no para producción. Más adelante existen versiones mucho más eficientes con librerías de optimización convexa.

Todo el código de este cuaderno funciona directamente en Google Colab: solo necesitas NumPy, Matplotlib y scikit-learn (esta última solo para generar el conjunto de puntos de ejemplo), que ya vienen instalados.

## El esqueleto de la clase SVM

Imagina que estás colgando un cuadro en la pared. No calculas la posición exacta con una fórmula: lo sujetas, das un paso atrás, compruebas si está torcido, ajustas un poco, vuelves a comprobar y repites hasta que encaja. El algoritmo que vas a programar hace exactamente eso con el vector `w`: prueba un valor, comprueba si cumple una condición sobre todos los puntos y, si no la cumple, lo ajusta y prueba otra vez.

Antes de llegar ahí, fíjate en la estructura básica que va a tener la clase. Toda clase en Python que representa un modelo suele tener esta forma: un método que la inicializa, uno que la entrena y uno que predice.

```python
class SVM:
    def __init__(self, visualizacion=True):
        self.visualizacion = visualizacion
        self.colores = {1: 'r', -1: 'b'}
        if self.visualizacion:
            self.fig = plt.figure()
            self.ax = self.fig.add_subplot(1, 1, 1)
        self.w = None
        self.b = None

    def fit(self, data):
        # Aquí va el algoritmo que busca los mejores w y b
        pass

    def predict(self, features):
        return np.sign(np.dot(np.array(features), self.w) + self.b)

    def visualizar(self):
        # Aquí va el código que dibuja los puntos, el hiperplano y los márgenes
        pass
```

Fíjate en los atributos `w` y `b`: son el corazón de todo lo que viene. `w` es el vector perpendicular al hiperplano y `b` es el término que lo desplaza respecto al origen. Mientras valgan `None`, la clase no sabe clasificar nada. El trabajo del método `fit` es rellenarlos con los valores correctos.

Los datos de entrada tienen una forma concreta: un diccionario con dos claves, `-1` y `1`, una por cada clase, cuyos valores son arrays de puntos. Más abajo generas ese diccionario a partir de un conjunto de puntos creado con `make_blobs`, en lugar de escribirlo a mano como haría alguien con un dataset diminuto.

Con esta estructura clara, toca completar la pieza que falta: el entrenamiento.

## Entrenar el SVM: encontrar w y b paso a paso

Recuerda la restricción que debe cumplir cada punto para estar bien clasificado con margen suficiente:

```
yi * (w . xi + b) >= 1
```

Aquí `xi` es un punto del conjunto de datos, `yi` es su clase (uno o menos uno) y el producto `w . xi + b` es la distancia, con signo, de ese punto al hiperplano. Si la restricción se cumple para todos los puntos, tienes un candidato válido de `w` y `b`. Entre todos los candidatos válidos, ¿cuál eliges? El que tiene el `w` más pequeño en magnitud, porque un `w` pequeño corresponde a un margen grande.

La estrategia es sencilla de describir: parte de un `w` grande y ve reduciéndolo poco a poco, comprobando en cada paso si la restricción se cumple para todos los puntos. En cuanto deja de cumplirse, sabes que el valor anterior era el límite, y ese es tu candidato.

Para no perder precisión ni tiempo, el algoritmo usa pasos de tamaño decreciente: empieza con pasos grandes (10% del valor máximo de las características), sigue con pasos medianos (1%) y termina con pasos pequeños (0,1%). Es la misma idea que usarías para encajar un tornillo: primero lo giras rápido y, cuando notas resistencia, giras poco a poco para no pasarte.

El sesgo `b` se explora de forma parecida, pero con pasos más grandes, porque no necesita tanta precisión como `w`, y afinarlo demasiado sale caro en tiempo de cálculo.

¿Ya intuyes por qué este método no escala bien a datasets grandes? Cada combinación de `w` y `b` que pruebas obliga a recorrer todos los puntos del conjunto de datos. Con pocos puntos, como los que usas hoy, tarda solo una fracción de segundo; con miles de puntos, se vuelve lento enseguida.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs


class SVM:
    def __init__(self, visualizacion=True):
        # Guarda si quieres ver el resultado en un gráfico
        self.visualizacion = visualizacion
        # Un color para cada clase: rojo para la clase 1, azul para la -1
        self.colores = {1: 'r', -1: 'b'}

        if self.visualizacion:
            self.fig = plt.figure()
            self.ax = self.fig.add_subplot(1, 1, 1)

        # Todavía no conocemos el vector de pesos ni el sesgo
        self.w = None
        self.b = None

    def fit(self, data):
        self.data = data
        # Diccionario para guardar candidatos: {magnitud de w: [w, b]}
        opt_dict = {}

        # Cuatro combinaciones de signos para probar w en todas las direcciones
        transformaciones = [[1, 1], [-1, 1], [-1, -1], [1, -1]]

        # Buscamos el valor máximo y mínimo entre todas las características
        todos_los_valores = []
        for clase in self.data:
            for punto in self.data[clase]:
                for valor in punto:
                    todos_los_valores.append(valor)

        self.max_valor = max(todos_los_valores)
        self.min_valor = min(todos_los_valores)
        todos_los_valores = None  # liberamos memoria, ya no la necesitamos

        # Pasos de tamaño decreciente: grandes, medianos y pequeños
        pasos = [self.max_valor * 0.1,
                 self.max_valor * 0.01,
                 self.max_valor * 0.001]

        # b se explora con pasos más grandes: no necesita tanta precisión
        b_rango_multiplo = 5
        b_paso_multiplo = 5
        ultimo_optimo = self.max_valor * 10

        for paso in pasos:
            w = np.array([ultimo_optimo, ultimo_optimo])
            optimizado = False

            while not optimizado:
                for b in np.arange(-1 * self.max_valor * b_rango_multiplo,
                                    self.max_valor * b_rango_multiplo,
                                    paso * b_paso_multiplo):
                    for transformacion in transformaciones:
                        w_candidato = w * transformacion
                        cumple_restriccion = True

                        # Comprobamos la restricción en cada punto de cada clase
                        for clase in self.data:
                            for punto in self.data[clase]:
                                yi = clase
                                if not yi * (np.dot(w_candidato, punto) + b) >= 1:
                                    cumple_restriccion = False
                                    break
                            if not cumple_restriccion:
                                break

                        if cumple_restriccion:
                            opt_dict[np.linalg.norm(w_candidato)] = [w_candidato, b]

                if w[0] < 0:
                    # Ya no tiene sentido seguir reduciendo w en este paso
                    optimizado = True
                else:
                    w = w - paso

            # Nos quedamos con el candidato de menor magnitud: el mejor margen
            normas = sorted([n for n in opt_dict])
            eleccion = opt_dict[normas[0]]
            self.w = eleccion[0]
            self.b = eleccion[1]
            ultimo_optimo = eleccion[0][0] + paso * 2

    def predict(self, features):
        clasificacion = np.sign(np.dot(np.array(features), self.w) + self.b)
        if clasificacion != 0 and self.visualizacion:
            self.ax.scatter(features[0], features[1], s=200, marker='*',
                             c=self.colores[clasificacion])
        return clasificacion

    def visualizar(self):
        # Dibuja todos los puntos de entrenamiento, coloreados por clase
        [[self.ax.scatter(punto[0], punto[1], s=100, color=self.colores[clase])
          for punto in self.data[clase]] for clase in self.data]

        def hiperplano(x, w, b, desplazamiento):
            return (-w[0] * x - b + desplazamiento) / w[1]

        # Un poco de margen extra a los lados del gráfico
        rango_datos = (self.min_valor * 0.9, self.max_valor * 1.1)
        x_min = rango_datos[0]
        x_max = rango_datos[1]

        # Margen positivo: w.x + b = 1
        psv1 = hiperplano(x_min, self.w, self.b, 1)
        psv2 = hiperplano(x_max, self.w, self.b, 1)
        self.ax.plot([x_min, x_max], [psv1, psv2], 'k')

        # Margen negativo: w.x + b = -1
        nsv1 = hiperplano(x_min, self.w, self.b, -1)
        nsv2 = hiperplano(x_max, self.w, self.b, -1)
        self.ax.plot([x_min, x_max], [nsv1, nsv2], 'k')

        # Hiperplano de decisión: w.x + b = 0
        db1 = hiperplano(x_min, self.w, self.b, 0)
        db2 = hiperplano(x_max, self.w, self.b, 0)
        self.ax.plot([x_min, x_max], [db1, db2], 'y--')

        plt.show()

## Lee el bucle con calma

Dentro de `fit` hay cuatro niveles de bucles anidados, y conviene que los identifiques antes de seguir:

- el bucle exterior recorre los tamaños de paso, de grande a pequeño
- para cada paso, el bucle `while` reduce `w` hasta que deja de cumplir la condición de parada
- dentro, recorres valores posibles de `b`
- y para cada `b`, pruebas las cuatro combinaciones de signos de `w`

Cada combinación que supera la comprobación en todos los puntos entra en `opt_dict` junto a su magnitud. Al terminar cada tanda de pasos, ordenas esas magnitudes y te quedas con la más pequeña: ese es tu mejor `w` y `b` hasta el momento. El resultado alimenta el siguiente tramo de pasos, cada vez más fino.

¿Y `predict`? Ahí la parte difícil ya pasó. Aplicas la fórmula que ya conoces del capítulo anterior, `signo(w . x + b)`, y el signo del resultado te dice a qué clase pertenece el punto. Cuando la visualización está activa, además, cada punto predicho se marca en el gráfico con una estrella, del color de la clase asignada.

Para dibujar el hiperplano y los márgenes, `visualizar` usa una función auxiliar, `hiperplano`, que, dada una coordenada `x`, calcula la `y` correspondiente sobre una de tres líneas: el margen positivo (`w.x + b = 1`), el margen negativo (`w.x + b = -1`) o el propio hiperplano de decisión (`w.x + b = 0`). Despejando la `y` de esa ecuación obtienes esa función.

## Prueba tu SVM con un conjunto de puntos generado con make_blobs

En vez de escribir los puntos a mano, genera un conjunto pequeño con `make_blobs`, la misma función de scikit-learn que ya usaste para crear conjuntos de prueba en clases anteriores. Le pides dos centros, así obtienes dos grupos de puntos bien diferenciados, ideales para un primer SVM.

`make_blobs` etiqueta los grupos con 0 y 1, pero tu clase SVM espera las etiquetas -1 y 1. Por eso, antes de entrenar, conviertes las etiquetas y organizas los puntos en el diccionario que espera `fit`.

In [ ]:
X, y = make_blobs(n_samples=10, centers=2, n_features=2, cluster_std=0.8,
                   center_box=(2, 10), random_state=1)

# La clase SVM espera un diccionario {-1: puntos, 1: puntos}
datos = {
    -1: X[y == 0],
    1: X[y == 1]
}

print("Puntos de la clase -1:")
print(datos[-1])
print("\nPuntos de la clase 1:")
print(datos[1])

Con el diccionario listo, entrena el modelo y comprueba cómo clasifica algunos puntos nuevos. Recuerda que `fit` no usa descenso de gradiente: recorre valores de `w` y `b` probando la restricción sobre estos diez puntos, así que puede tardar un par de segundos.

In [ ]:
svm = SVM()
svm.fit(datos)

print("w =", svm.w)
print("b =", svm.b)

puntos_a_predecir = [[6.5, 7], [7, 6], [2, 4], [1.5, 5], [4.5, 5.5]]

for punto in puntos_a_predecir:
    print(punto, svm.predict(punto))

Si ejecutas la celda anterior, ves que los dos primeros puntos, cercanos al grupo de arriba a la derecha, se clasifican como -1, y los dos siguientes, cercanos al grupo de abajo a la izquierda, se clasifican como 1. Fíjate en el último, `[4.5, 5.5]`: está casi a medio camino entre ambos grupos, y aun así cae de un lado concreto, el que le corresponde según en qué lado del hiperplano queda exactamente.

## Dibuja el hiperplano y los márgenes

Con `w` y `b` ya calculados, solo falta ver el resultado. Llama a `visualizar` y observa el gráfico completo: los puntos de entrenamiento, las estrellas de las predicciones y las tres líneas paralelas.

In [ ]:
svm.visualizar()

En el gráfico ves los puntos azules, de la clase -1, y los puntos rojos, de la clase 1, cada uno agrupado en su propia esquina. La línea amarilla discontinua es el hiperplano de decisión, y las dos líneas negras continuas, a ambos lados, son los márgenes. Las estrellas son los cinco puntos que acabas de predecir, coloreadas según la clase que les asignó el modelo. Comprueba que cada estrella cae del lado que corresponde según su posición.

Este proceso es costoso porque revisa todos los puntos del dataset en cada combinación de `w` y `b` que prueba. Con los diez puntos de hoy, tarda una fracción de segundo; con datasets grandes, se vuelve lento enseguida. Ahí es donde métodos de optimización más avanzados marcan la diferencia, algo que verás más adelante.

## Resumen: lo que te llevas de esta clase

- La clase SVM necesita solo tres piezas: `fit` para entrenar, `predict` para clasificar y una visualización para interpretar el resultado
- Los atributos `w` y `b` empiezan vacíos, y `fit` es la única función responsable de calcularlos
- El entrenamiento que programaste no usa descenso de gradiente ni librerías de optimización: prueba valores de `w` cada vez más pequeños y los descarta en cuanto algún punto incumple la restricción `yi * (w . xi + b) >= 1`
- Entre todos los candidatos válidos de `w`, el algoritmo elige siempre el de menor magnitud, porque corresponde al margen más amplio
- `predict` aplica la misma fórmula del capítulo anterior, `signo(w . x + b)`, para decidir la clase de un punto nuevo
- La visualización dibuja el hiperplano de decisión y sus dos márgenes despejando la `y` de la ecuación `w.x + b = valor`, con `valor` igual a 0, 1 o -1

**Ahora te toca a ti.** Abre el cuaderno de práctica de esta clase y completa el método `predict` y la visualización de otro SVM, entrenado sobre un conjunto de puntos distinto.

**Siglas usadas en esta clase:** SVM (Support Vector Machine, máquina de vectores de soporte).